Day 2 done. Neural bigram loss ≈ probabilities of counts = 2.4544. Next: Day 3, micrograd cleanup.

In [1]:
import torch

In [2]:
names = open('names.txt','r').read().splitlines()

In [3]:
len(names)

32033

In [4]:
b = {}
bigram = ()
for name in names:
  chr_seq = ['.']+list(name)+['.']
  for ch1,ch2 in zip(chr_seq,chr_seq[1:]):
    bigram=(ch1,ch2)
    b[bigram] = b.get(bigram,0)+1

In [5]:
sorted_b= sorted(b.items(),key=lambda x:x[1],reverse=True)
sorted_b[:5]

[(('n', '.'), 6763),
 (('a', '.'), 6640),
 (('a', 'n'), 5438),
 (('.', 'a'), 4410),
 (('e', '.'), 3983)]

In [6]:
alphas = ''
for name in names:
  alphas = alphas+name.strip()
vocab = sorted(list(set(alphas)))
atoi = {a:i+1 for i,a in enumerate(vocab)}
atoi['.'] = 0
itoa = {i:a for a,i in atoi.items()}
N = torch.zeros([27,27],dtype=int)
N.shape

torch.Size([27, 27])

In [7]:
counts=0
for x in b.items():
  ch1 = x[0][0]
  ch2 = x[0][1]
  idx1 = atoi[ch1]
  idx2 = atoi[ch2]
  N[idx1][idx2] += x[1]
  counts+=x[1]
counts

228146

Broadcasting to convert the N vector into a matrix of probabilities

In [8]:
P = (N+1).float() # 1 is the bias here Added to avoid data sparsing
P = P/P.sum(1,keepdim=True)

Probability Distribution for the first character of a name

In [9]:
g = torch.Generator().manual_seed(247483647)
idx = torch.multinomial(P[0],1,replacement=True,generator=g).item()
idx

5

In [10]:
g = torch.Generator().manual_seed(134)
for i in range(5):
  out = []
  idx = 0
  while (True):
    p = P[idx]
    idx = torch.multinomial(p,1,replacement=True,generator=g).item() #Also updates the idx variable to the character predicted from the probabilities
    out.append(itoa[idx])
    if idx==0:# Breaks whenever it sees a full stop predicted again
      break
  print(''.join(out))

andele.
judelishalix.
tan.
adeetah.
lorlyli.


Now we want to calculate and maximize the likelihood, which is same as maximizing the log likelihood and minimizing the negative log likelihood also known as the total Loss.

In [11]:
log_likelihood = 0.0
n = 0

for name in names:
  ch_seq = ['.']+list(name)+['.']
  for chr1,chr2 in zip(ch_seq,ch_seq[1:]):
    id1 = atoi[chr1]
    id2 = atoi[chr2]
    prob = P[id1][id2]
    logprob=torch.log(prob)
    log_likelihood+=logprob
    n+=1

nll = -log_likelihood
print(f'Log Likelihood = {log_likelihood}')
print(f'Negative Log Likelihood = {nll}')
print(f'Loss/Mean Negative Log Likelihood = {nll/n}')

Log Likelihood = -559951.5625
Negative Log Likelihood = 559951.5625
Loss/Mean Negative Log Likelihood = 2.4543561935424805


Now we'll try to minimize the loss or the mean nll using the neural network structure

In [12]:
inputs = []
labels = []

for name in names:
  chr_seq = ['.']+list(name)+['.']
  for c1,c2 in zip(chr_seq,chr_seq[1:]):
    inputs.append(atoi[c1])
    labels.append(atoi[c2])

inputs = torch.tensor(inputs)
labels = torch.tensor(labels)
num = inputs.nelement()

Converting to One Hot Encoding to be input to the neural nets

In [13]:
from torch.nn import functional as F
xenc = F.one_hot(inputs,num_classes=27).float() # Shape = ([228146, 27])
yenc = F.one_hot(labels,num_classes=27).float()

In [24]:
yenc.shape

torch.Size([228146, 27])

In [14]:
W = torch.randn((27,27),generator = g, requires_grad = True) #Weights Matrix

xenc is one-hot, so xenc @ W gives me row i(the input character's index) of W, because every other entry is multiplied by 0

In [15]:
logits = xenc @ W
loss = F.cross_entropy(logits,labels)
loss.item()

3.677241563796997

Training

In [18]:
for k in range(200):
  logits = xenc@W
  loss = F.cross_entropy(logits,labels)

  W.grad = None
  loss.backward()

  W.data+=-35*W.grad

  if k%10==0:
    print(f' {k} : {loss.item()}')

print("Final Loss = ",loss.item())

 0 : 2.462674379348755
 10 : 2.4623281955718994
 20 : 2.4620065689086914
 30 : 2.4617068767547607
 40 : 2.4614267349243164
 50 : 2.461164951324463
 60 : 2.46091890335083
 70 : 2.460688352584839
 80 : 2.460470676422119
 90 : 2.460265636444092
 100 : 2.4600722789764404
 110 : 2.4598891735076904
 120 : 2.4597151279449463
 130 : 2.4595510959625244
 140 : 2.459394931793213
 150 : 2.4592463970184326
 160 : 2.4591052532196045
 170 : 2.458970546722412
 180 : 2.4588427543640137
 190 : 2.4587204456329346
Final Loss =  2.458615303039551


In [19]:
g = torch.Generator().manual_seed(134)
for _ in range(5):
    out, ix = [], 0
    while True:
        xenc_i = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc_i @ W
        p = F.softmax(logits, dim=1)
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        if ix == 0:
            break
        out.append(itoa[ix])
    print(''.join(out))

andele
judelishalix
tan
adeetah
lorlyli
